In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Conv2D, GlobalAveragePooling2D, MaxPooling2D, Dropout, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [7]:
data_dir = "/kaggle/input/datasets/iamsouravbanerjee/animal-image-dataset-90-different-animals/animals/animals"

train_ds = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2, subset="training", seed=42, image_size=(224, 224), batch_size=32, label_mode='categorical')
valid_ds = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2, subset="validation", seed=42, image_size=(224, 224), batch_size=32, label_mode='categorical')

Found 5400 files belonging to 90 classes.
Using 4320 files for training.


I0000 00:00:1792515104.914921      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792515104.918850      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 5400 files belonging to 90 classes.
Using 1080 files for validation.


In [8]:
class_names = train_ds.class_names
num_classes = len(class_names)
print("Total Classes:", num_classes)

Total Classes: 90


In [9]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
valid_ds = valid_ds.prefetch(AUTOTUNE)

In [10]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
])

In [11]:
model = Sequential()

model.add(tf.keras.Input(shape=(224, 224, 3)))
model.add(data_augmentation)

model.add(Conv2D(32, (3,3), activation='relu'))
model.add(MaxPooling2D())

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D())

model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D())

model.add(Conv2D(256, (3,3), activation='relu'))
model.add(MaxPooling2D())


model.add(GlobalAveragePooling2D())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(90, activation='softmax'))

In [12]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [13]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "AnimalPrediction.keras",
    monitor='val_accuracy',
    save_best_only=True
)

In [14]:
history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=25,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 100s 696ms/step - accuracy: 0.0176 - loss: 4.9043 - val_accuracy: 0.0231 - val_loss: 4.3323
Epoch 2/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 10s 76ms/step - accuracy: 0.0347 - loss: 4.3207 - val_accuracy: 0.0306 - val_loss: 4.2211
Epoch 3/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 10s 76ms/step - accuracy: 0.0412 - loss: 4.2161 - val_accuracy: 0.0407 - val_loss: 4.2234
Epoch 4/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 10s 77ms/step - accuracy: 0.0493 - loss: 4.1474 - val_accuracy: 0.0528 - val_loss: 4.1548
Epoch 5/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 10s 76ms/step - accuracy: 0.0535 - loss: 4.1005 - val_accuracy: 0.0667 - val_loss: 4.0958
Epoch 6/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 10s 76ms/step - accuracy: 0.0551 - loss: 4.0558 - val_accuracy: 0.0639 - val_loss: 4.0795
Epoch 7/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 10s 77ms/step - accuracy: 0.0713 - loss: 3.9932 - val_accuracy: 0.0796 - val_loss: 3.9763
Epoch 8/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 10s 75ms/step - accuracy: 0.0794 - loss: 3.9178 

In [15]:
loss, accuracy = model.evaluate(valid_ds)

print("Validation Accuracy:", accuracy)

34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 59ms/step - accuracy: 0.2167 - loss: 3.3357
Validation Accuracy: 0.21666666865348816


# Pre-Train Model

In [16]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = Input(shape=(224, 224, 3))

x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(num_classes, activation="softmax")(x)

model = Model(inputs, outputs)
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 90)             │        11,610 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,433,562 (9.28 MB)

 Trainable params: 175,578 (685.85 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [17]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [18]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint("PreTrainAnimalPrediction.keras", monitor='val_accuracy', save_best_only=True)

In [19]:
history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=25,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 19s 97ms/step - accuracy: 0.2338 - loss: 3.4946 - val_accuracy: 0.6602 - val_loss: 1.7505
Epoch 2/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 11s 82ms/step - accuracy: 0.5586 - loss: 1.8058 - val_accuracy: 0.7306 - val_loss: 1.0423
Epoch 3/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 11s 81ms/step - accuracy: 0.6539 - loss: 1.3739 - val_accuracy: 0.7657 - val_loss: 0.8444
Epoch 4/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 11s 85ms/step - accuracy: 0.6866 - loss: 1.1610 - val_accuracy: 0.7944 - val_loss: 0.7499
Epoch 5/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 11s 78ms/step - accuracy: 0.7222 - loss: 1.0167 - val_accuracy: 0.7917 - val_loss: 0.6827
Epoch 6/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 11s 83ms/step - accuracy: 0.7495 - loss: 0.9374 - val_accuracy: 0.8102 - val_loss: 0.6343
Epoch 7/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 11s 82ms/step - accuracy: 0.7644 - loss: 0.8537 - val_accuracy: 0.8204 - val_loss: 0.6097
Epoch 8/25
135/135 ━━━━━━━━━━━━━━━━━━━━ 11s 79ms/step - accuracy: 0.7764 - loss: 0.8182 - 

In [20]:
loss, accuracy = model.evaluate(valid_ds)

print("Validation Accuracy:", accuracy)

34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 59ms/step - accuracy: 0.8398 - loss: 0.5475
Validation Accuracy: 0.8398148417472839


# Prediction

In [21]:
import os
import numpy as np
import tensorflow as tf

model = tf.keras.models.load_model("PreTrainAnimalPrediction.keras")

data_dir = "/kaggle/input/datasets/iamsouravbanerjee/animal-image-dataset-90-different-animals/animals/animals"
class_names = sorted(os.listdir(data_dir))

def predict_image(path):
    img = tf.keras.utils.load_img(path, target_size=(224, 224))
    arr = tf.keras.utils.img_to_array(img) 
    arr = np.expand_dims(arr, axis=0)
    preds = model.predict(arr)
    idx = int(np.argmax(preds[0]))
    print(f"Prediction: {class_names[idx]} ({preds[0][idx] * 100:.2f}%)")

predict_image("/kaggle/input/datasets/iamsouravbanerjee/animal-image-dataset-90-different-animals/animals/animals/fox/046e05e07b.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Prediction: fox (99.93%)
